In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp03/ex_4.py ──
"""
Shared infrastructure for MLFP03 Exercise 4 — Gradient Boosting Deep Dive.

Contains: Singapore credit-scoring data loader, preprocessing pipeline
wrapper, numpy/polars conversion, model-zoo factories (XGBoost/LightGBM/
CatBoost with identical defaults), evaluation metric helpers, and the
output directory used by every technique file.

Technique-specific code (from-scratch boosting loops, split-gain formulas,
hyperparameter sweeps, early-stopping analysis) does NOT belong here — it
lives in the per-technique files under `modules/mlfp03/solutions/ex_4/`.
"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    log_loss,
    roc_auc_score,
)

from kailash_ml import PreprocessingPipeline
from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# CONFIG — output directory, random seeds, dataset tag
# ════════════════════════════════════════════════════════════════════════

SEED = 42
DATASET_MODULE = "mlfp02"
DATASET_FILE = "sg_credit_scoring.parquet"
TARGET_COLUMN = "default"

OUTPUT_DIR = Path("outputs") / "mlfp03" / "ex_4_boosting"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore credit-scoring, shared across all four techniques
# ════════════════════════════════════════════════════════════════════════


def load_credit_data() -> pl.DataFrame:
    """Load the Singapore credit-scoring dataset via MLFPDataLoader."""
    loader = MLFPDataLoader()
    return loader.load(DATASET_MODULE, DATASET_FILE)


def prepare_credit_split() -> dict[str, Any]:
    """Load credit data and return a train/test split ready for boosting.

    Returns a dict with:
      X_train, y_train, X_test, y_test : numpy arrays
      feature_names                    : list[str]
      default_rate                     : float (positive-class prevalence)

    Tree models do not need normalisation, so we set ``normalize=False``.
    Categoricals are ordinal-encoded because XGBoost/LightGBM expect numeric
    input; CatBoost would accept raw categoricals but we keep the pipeline
    consistent across all three libraries for a fair comparison.
    """
    credit = load_credit_data()

    pipeline = PreprocessingPipeline()
    result = pipeline.setup(
        data=credit,
        target=TARGET_COLUMN,
        train_size=0.8,
        seed=SEED,
        normalize=False,
        categorical_encoding="ordinal",
        imputation_strategy="median",
    )

    feature_cols = [c for c in result.train_data.columns if c != TARGET_COLUMN]
    X_train, y_train, col_info = to_sklearn_input(
        result.train_data,
        feature_columns=feature_cols,
        target_column=TARGET_COLUMN,
    )
    X_test, y_test, _ = to_sklearn_input(
        result.test_data,
        feature_columns=feature_cols,
        target_column=TARGET_COLUMN,
    )

    return {
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test,
        "feature_names": col_info["feature_columns"],
        "default_rate": float(credit[TARGET_COLUMN].mean()),
    }


# ════════════════════════════════════════════════════════════════════════
# MODEL FACTORIES — identical defaults for fair comparison
# ════════════════════════════════════════════════════════════════════════


def make_xgboost(
    n_estimators: int = 500,
    learning_rate: float = 0.1,
    max_depth: int = 6,
    **extra: Any,
):
    """Build an XGBoost classifier with course-standard defaults."""
    import xgboost as xgb

    return xgb.XGBClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        eval_metric="logloss",
        random_state=SEED,
        verbosity=0,
        **extra,
    )


def make_lightgbm(
    n_estimators: int = 500,
    learning_rate: float = 0.1,
    max_depth: int = 6,
    **extra: Any,
):
    """Build a LightGBM classifier with course-standard defaults."""
    import lightgbm as lgb

    return lgb.LGBMClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        num_leaves=31,
        random_state=SEED,
        verbose=-1,
        **extra,
    )


def make_catboost(
    iterations: int = 500,
    learning_rate: float = 0.1,
    depth: int = 6,
    **extra: Any,
):
    """Build a CatBoost classifier with course-standard defaults."""
    import catboost as cb

    return cb.CatBoostClassifier(
        iterations=iterations,
        learning_rate=learning_rate,
        depth=depth,
        random_seed=SEED,
        verbose=0,
        **extra,
    )


# ════════════════════════════════════════════════════════════════════════
# EVALUATION — shared metric helpers for imbalanced binary classification
# ════════════════════════════════════════════════════════════════════════


def evaluate_classifier(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    threshold: float = 0.5,
) -> dict[str, float]:
    """Return the full boosting-eval metric bundle.

    AUC-PR is the primary metric — with a 12% default rate, AUC-ROC rewards
    models that rank negatives correctly even when they miss every default.
    """
    y_pred = (y_proba >= threshold).astype(int)
    return {
        "auc_roc": float(roc_auc_score(y_true, y_proba)),
        "auc_pr": float(average_precision_score(y_true, y_proba)),
        "log_loss": float(log_loss(y_true, y_proba)),
        "brier": float(brier_score_loss(y_true, y_proba)),
        "f1": float(f1_score(y_true, y_pred)),
    }


def print_metrics(
    name: str, metrics: dict[str, float], train_time: float | None = None
) -> None:
    """One-line headline: AUC-ROC | AUC-PR | Log Loss | F1 | Time."""
    time_str = f" | time={train_time:.2f}s" if train_time is not None else ""
    print(
        f"  {name}: "
        f"AUC-ROC={metrics['auc_roc']:.4f} | "
        f"AUC-PR={metrics['auc_pr']:.4f} | "
        f"log_loss={metrics['log_loss']:.4f} | "
        f"F1={metrics['f1']:.4f}"
        f"{time_str}"
    )


# ════════════════════════════════════════════════════════════════════════
# FROM-SCRATCH GRADIENT BOOSTING (used by 01_boosting_theory.py)
# ════════════════════════════════════════════════════════════════════════


def xgb_split_gain(
    g_left: float,
    h_left: float,
    g_right: float,
    h_right: float,
    lambda_reg: float = 1.0,
    gamma: float = 0.0,
) -> float:
    """XGBoost split-gain formula from 2nd-order Taylor expansion of log-loss.

    Gain = ½ [G_L²/(H_L+λ) + G_R²/(H_R+λ) - (G_L+G_R)²/(H_L+H_R+λ)] - γ
    """
    left_score = g_left**2 / (h_left + lambda_reg)
    right_score = g_right**2 / (h_right + lambda_reg)
    parent_score = (g_left + g_right) ** 2 / (h_left + h_right + lambda_reg)
    return 0.5 * (left_score + right_score - parent_score) - gamma


def make_1d_demo(n: int = 200) -> tuple[np.ndarray, np.ndarray]:
    """Generate a 1D logistic-shaped binary classification demo.

    Used by the from-scratch boosting loop to show that residual-fitting
    converges in just a handful of rounds.
    """
    rng = np.random.default_rng(SEED)
    x = rng.uniform(0, 1, n).reshape(-1, 1)
    true_proba = 1 / (1 + np.exp(-10 * (x.ravel() - 0.5)))
    y = rng.binomial(1, true_proba)
    return x, y




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 4.4: Boosting Tuning — Sweeps, Heatmaps, Early Stopping
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Sweep learning rate and see how AUC-PR changes across η
#   - Build a learning_rate × max_depth heatmap and read the interaction
#   - Use early stopping to pick n_estimators automatically instead of
#     guessing
#   - Explain why "grid search over independent dials" is the wrong
#     mental model for boosting (Exercise 7 will replace it with Bayesian)
#   - Produce a final production-ready configuration
#
# PREREQUISITES: Exercise 4.2/4.3 (XGBoost and LightGBM/CatBoost).
#
# ESTIMATED TIME: ~40 min
#
# TASKS:
#   1. Theory — hyperparameter interaction, why grids lie
#   2. Build — a learning-rate sweep and a 2-D depth×η heatmap
#   3. Train — run the sweep; run early stopping with a 2000-round budget
#   4. Visualise — heatmap + LR sweep + early-stopping curve
#   5. Apply — Grab credit pre-approval team tunes for business metric
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go
from dotenv import load_dotenv
from sklearn.metrics import average_precision_score


load_dotenv()



## THEORY — Hyperparameter Interaction Is Why Grid Search Lies


In [ ]:
# The naive approach to tuning boosting is "vary one knob at a time":
# hold max_depth fixed at 6, sweep learning_rate, pick the best, then
# hold learning_rate fixed and sweep max_depth. This is wrong because
# the knobs INTERACT.
#
# Example of interaction:
#   - At learning_rate=0.01, max_depth=10 is fine — the small step size
#     keeps the deep trees from overfitting.
#   - At learning_rate=0.2, max_depth=10 overfits badly — each big step
#     drives a deep tree far past the optimal loss.
#
# The right mental model is a 2-D surface over (η, depth). A 1-D sweep
# only sees a slice of that surface and can lead you to a local optimum
# that doesn't hold once the other knob moves.
#
# Two practical tools replace naive grid search:
#
#   1. Early stopping: set n_estimators to a large budget (2000+), let
#      validation loss tell you when to stop. This turns n_estimators
#      from a tuning knob into a self-tuning parameter.
#
#   2. 2-D heatmap of (learning_rate × max_depth). Small grids (3x4)
#      are enough to SEE the interaction surface. Beyond 2 dimensions,
#      grids explode combinatorially — Exercise 7 uses Bayesian
#      optimisation to scale this up.



## TASK 2 — BUILD the sweep grids


In [ ]:
print("\n" + "=" * 70)
print("  Boosting Hyperparameter Tuning on Singapore Credit")
print("=" * 70)

data = prepare_credit_split()
X_train, y_train = data["X_train"], data["y_train"]
X_test, y_test = data["X_test"], data["y_test"]

print(f"\n  Train: {X_train.shape} | Test: {X_test.shape}")
print(f"  Default rate: {data['default_rate']:.2%}")

learning_rates = [0.01, 0.03, 0.05, 0.1, 0.2, 0.5]
depths_sweep = [3, 5, 6, 8, 10]
lr_sweep_for_heatmap = [0.01, 0.05, 0.1, 0.2]



## TASK 3 — TRAIN the sweeps


In [ ]:
# --- 3a. 1-D learning-rate sweep (XGBoost, depth fixed at 6) ------------
print("\n  --- Learning-rate sweep (XGBoost, depth=6, 500 rounds) ---")
print(f"  {'lr':>6}  {'AUC-ROC':>10}  {'AUC-PR':>10}")
print("  " + "─" * 32)

lr_curve: list[tuple[float, float, float]] = []  # (lr, auc_roc, auc_pr)
for lr in learning_rates:
    m = make_xgboost(n_estimators=500, learning_rate=lr, max_depth=6)
    m.fit(X_train, y_train, verbose=False)
    y_p = m.predict_proba(X_test)[:, 1]
    auc_roc = float(average_precision_score(y_test, y_p))  # fallback if roc breaks
    metrics = evaluate_classifier(y_test, y_p)
    lr_curve.append((lr, metrics["auc_roc"], metrics["auc_pr"]))
    print(f"  {lr:>6.2f}  {metrics['auc_roc']:>10.4f}  {metrics['auc_pr']:>10.4f}")


# --- 3b. 2-D learning_rate × max_depth heatmap --------------------------
print("\n  --- Heatmap: learning_rate × max_depth (XGBoost, 300 rounds) ---")
print(f"  {'':>8}", end="")
for d in depths_sweep:
    print(f"  d={d:<3}   ", end="")
print()
print("  " + "─" * (10 + 10 * len(depths_sweep)))

heatmap = np.zeros((len(lr_sweep_for_heatmap), len(depths_sweep)))
for i, lr in enumerate(lr_sweep_for_heatmap):
    print(f"  lr={lr:<5}", end="")
    for j, d in enumerate(depths_sweep):
        m = make_xgboost(n_estimators=300, learning_rate=lr, max_depth=d)
        m.fit(X_train, y_train, verbose=False)
        y_p = m.predict_proba(X_test)[:, 1]
        auc_pr = float(average_precision_score(y_test, y_p))
        heatmap[i, j] = auc_pr
        print(f"  {auc_pr:>8.4f}", end="")
    print()

best_idx = np.unravel_index(heatmap.argmax(), heatmap.shape)
best_lr = lr_sweep_for_heatmap[best_idx[0]]
best_depth = depths_sweep[best_idx[1]]
print(
    f"\n  Best combo: lr={best_lr}, depth={best_depth} "
    f"(AUC-PR={heatmap[best_idx]:.4f})"
)


# --- 3c. Early stopping (XGBoost, 2000-round budget) --------------------
print("\n  --- Early stopping (XGBoost, 2000-round budget, η=0.05) ---")
es_model = make_xgboost(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=6,
    early_stopping_rounds=50,
)
es_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
best_iter_xgb = int(es_model.best_iteration)
es_metrics = evaluate_classifier(y_test, es_model.predict_proba(X_test)[:, 1])
print(f"  XGBoost: best_iteration = {best_iter_xgb}/2000")
print_metrics("XGB+ES", es_metrics)

# Also early-stop LightGBM so students see it's the same idea, different API
import lightgbm as lgb  # local import keeps top imports clean

lgb_es = make_lightgbm(n_estimators=2000, learning_rate=0.05, max_depth=6)
lgb_es.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)
best_iter_lgb = int(lgb_es.best_iteration_)
lgb_es_metrics = evaluate_classifier(y_test, lgb_es.predict_proba(X_test)[:, 1])
print(f"  LightGBM: best_iteration = {best_iter_lgb}/2000")
print_metrics("LGB+ES", lgb_es_metrics)



### Checkpoint 1


In [ ]:
assert len(lr_curve) == len(learning_rates), "Every learning rate must be evaluated"
assert heatmap.shape == (len(lr_sweep_for_heatmap), len(depths_sweep))
assert heatmap.max() > 0, "At least one heatmap cell must have positive AUC-PR"
assert best_iter_xgb < 2000, "XGBoost early stopping must fire before the budget"
assert best_iter_lgb < 2000, "LightGBM early stopping must fire before the budget"
# INTERPRETATION: Early stopping fires well before 2000 rounds in almost
# every production setting. That's the signal the 2000-round budget was
# safely high enough — if best_iteration hits 2000, raise the budget.
print("\n[ok] Checkpoint 1 passed — sweeps + heatmap + early stopping all ran\n")



## TASK 4 — VISUALISE the tuning surface


In [ ]:
# 4a. LR curve
lr_fig = go.Figure()
lrs = [row[0] for row in lr_curve]
auc_prs = [row[2] for row in lr_curve]
lr_fig.add_trace(
    go.Scatter(x=lrs, y=auc_prs, mode="lines+markers", name="XGBoost AUC-PR")
)
lr_fig.update_layout(
    title="Learning-Rate Sensitivity — XGBoost on Singapore Credit",
    xaxis_title="learning_rate (η)",
    yaxis_title="AUC-PR",
    xaxis=dict(type="log"),
)
lr_path = OUTPUT_DIR / "ex4_04_lr_curve.html"
lr_fig.write_html(lr_path)
print(f"  Saved: {lr_path}")

# 4b. Heatmap
heat_fig = go.Figure(
    data=go.Heatmap(
        z=heatmap,
        x=[f"d={d}" for d in depths_sweep],
        y=[f"η={lr}" for lr in lr_sweep_for_heatmap],
        colorscale="Viridis",
        colorbar=dict(title="AUC-PR"),
    )
)
heat_fig.update_layout(
    title="Hyperparameter Interaction Heatmap — XGBoost (300 rounds)",
    xaxis_title="max_depth",
    yaxis_title="learning_rate",
)
heat_path = OUTPUT_DIR / "ex4_04_heatmap.html"
heat_fig.write_html(heat_path)
print(f"  Saved: {heat_path}")

# 4c. Early-stopping comparison (fixed 500 vs early stop)
es_fig = go.Figure()
es_fig.add_trace(
    go.Bar(
        name="Fixed 500 rounds",
        x=["XGBoost", "LightGBM"],
        y=[lr_curve[3][2], lgb_es_metrics["auc_pr"]],  # lr=0.1 row for XGB
    )
)
es_fig.add_trace(
    go.Bar(
        name="Early stopping (2000 budget)",
        x=["XGBoost", "LightGBM"],
        y=[es_metrics["auc_pr"], lgb_es_metrics["auc_pr"]],
    )
)
es_fig.update_layout(
    title="Early Stopping vs Fixed Rounds — AUC-PR on Singapore Credit",
    barmode="group",
    yaxis_title="AUC-PR",
)
es_path = OUTPUT_DIR / "ex4_04_early_stopping.html"
es_fig.write_html(es_path)
print(f"  Saved: {es_path}")



### Checkpoint 2


In [ ]:
assert es_metrics["auc_pr"] > 0, "Early-stopped XGBoost should have positive AUC-PR"
# INTERPRETATION: The three figures give you three reads on the tuning
# surface. The LR curve is 1-D (easy to reason about but misleading).
# The heatmap is 2-D (hyperparameter interaction visible). Early stopping
# is the operational pattern that replaces guessing n_estimators. In
# production, you combine them: tune (η, depth) on a 2-D heatmap with
# early stopping driving n_estimators.
print("\n[ok] Checkpoint 2 passed — all tuning visualisations saved\n")



## TASK 5 — APPLY: Grab Credit Pre-Approval Tuning


In [ ]:
# SCENARIO: Grab Financial Group (Singapore HQ, operating across SEA)
# runs a GrabFin credit pre-approval model that decides which of its
# ~25M active Grab users get a "You're pre-approved for S$X" nudge in
# the app. The model must:
#   1. Rank users by default risk so the top-X% get the offer
#   2. Maintain a calibrated probability so the offer amount can be
#      priced from expected loss
#   3. Re-train weekly as new repayment data arrives
#
# The tuning task is NOT "maximise AUC-PR" — it is:
#   "maximise expected net revenue per eligible user, subject to a
#    regulatory ceiling on default rate for the approved pool"
#
# Translating that to tuning knobs:
#   - learning_rate: lower (0.03) is better for calibration — small
#     steps give smoother probability estimates. 04_04_lr_curve.html
#     shows AUC-PR is relatively flat across 0.03-0.1, so the team
#     picks 0.03 on calibration grounds.
#   - max_depth: the heatmap shows depth 6-8 is the sweet spot. Deeper
#     trees (10) overfit; shallower (3) underfit. Team picks 6.
#   - n_estimators: early stopping with a 3000-round budget and 100-
#     round patience. best_iteration is typically 800-1200 on a full
#     re-train — no need to guess.
#   - reg_lambda (λ): raised from the default 1.0 to 5.0 to pull leaf
#     weights toward zero, which smooths calibration at a tiny AUC-PR
#     cost (~0.002).
#
# BUSINESS IMPACT: Grab pre-approves ~S$2.4B in credit lines per year.
# Moving from a calibration-naive "just maximise AUC-PR" model to a
# calibration-aware tuning produces ~10-15% lift in pool-level expected
# net revenue — roughly S$50-80M/year — because mis-priced offers
# (either too cheap for the risk or too expensive vs the competition)
# get corrected. The tuning-time cost is a few hours of compute per
# week. The key is that the TUNING LOSS FUNCTION is not AUC-PR — it's
# a business metric, and the hyperparameters are chosen against it.
#
# This is the pattern every production boosting team uses: tune for the
# business metric, not for the offline AUC. Exercise 7 will formalise
# this with Bayesian optimisation over a business-metric objective.



## REFLECTION


[x] Swept learning_rate and saw AUC-PR change across 6 values
  [x] Built a 2-D learning_rate × max_depth heatmap and read the
      interaction surface (best: lr={best_lr}, depth={best_depth})
  [x] Used early stopping to pick n_estimators automatically
      (XGBoost best_iter={best_iter_xgb}, LightGBM best_iter={best_iter_lgb})
  [x] Explained why grid search over independent dials is the wrong
      mental model and why Exercise 7 will use Bayesian optimisation
  [x] Mapped tuning knobs to a business metric using the Grab scenario

  KEY INSIGHT: Hyperparameters interact. Never tune one at a time.
  Always use early stopping for n_estimators. And always tune against
  the business objective — AUC-PR is a proxy, not a destination.

  NEXT: Exercise 5 (class imbalance and calibration). You've seen that
  12% positive rate is survivable with AUC-PR + good boosting. Exercise
  5 pushes into 1-2% positive rates where SMOTE, cost-sensitive
  learning, focal loss, and probability calibration become essential.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

